## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from haystack import Pipeline, component
from hayhooks import BasePipelineWrapper
from haystack.dataclasses import Document
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

from openai import OpenAI
from openai.types.create_embedding_response import CreateEmbeddingResponse

In [ ]:
from sem_rag.config.settings import *
from sem_rag.storage.qdrant import get_document_store

In [ ]:
INSTRUCTION = "Represent the user's input."

MMR_CANDIDATES = 50
TOP_K = 5

In [ ]:
client = OpenAI(base_url=VLLM.API_URL, api_key=VLLM.API_KEY)

qwen_qdrant = get_document_store(Qdrant.COLLECTION_QWEN, Qdrant.EMBEDDING_DIM_QWEN)

In [ ]:
@component
class QueryEmbedder():
    def __init__(self, model: str, api_base_url: str, api_key: str):
        self.model = model
        self.client = OpenAI(base_url=api_base_url, api_key=api_key)

    @component.output_types(embedding=list[float])
    def run(self, text: str = "", image_data_urls: list[str] | None = None):
        if image_data_urls:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [
                        {"type": "image_url", "image_url": {"url": image_data_urls[0]}},
                        {"type": "text", "text": ""},
                ]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]
        else:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [{"type": "text", "text": text}]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]

        response: CreateEmbeddingResponse = self.client.post(
            "/embeddings",
            cast_to=CreateEmbeddingResponse,
            body={
                "messages": messages,
                "model": self.model,
                "encoding_format": "float",
                "continue_final_message": True,
                "add_special_tokens": True,
            },
        )

        return {"embedding": response.data[0].embedding}

In [ ]:
@component
class MMRReranker():
    def __init__(self, top_k: int = 5, lambda_: float = 0.8):
        self.top_k = top_k
        self.lambda_ = lambda_

    @component.output_types(documents=list[Document])
    def run(
        self,
        query_embedding: list[float],
        documents: list[Document],
    ):
        if not documents:
            return {"documents": []}

        if len(documents) <= self.top_k:
            return {"documents": documents}

        # Turn embeddings into NumPy arrays
        query = np.asarray(query_embedding, dtype=np.float32)
        embeddings = np.asarray([document.embedding for document in documents], dtype=np.float32)

        # Normalize embeddings for cosine similarity
        query = query / np.linalg.norm(query)
        embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

        # Similarity between query and each candidate -> array with [sim(query, A), sim(query, B), ...]
        query_similarities = embeddings @ query

        # Pairwise candidate similarities -> 50x50 matrix with how similar every candidate is to every other candidate
        candidate_similarities = embeddings @ embeddings.T

        selected = [] # list of selected images' indexes
        remaining = list(range(len(documents))) # list of indexes [0, 1, 2, ...]

        # First result: index of the most similar image to query
        first = int(np.argmax(query_similarities))
        selected.append(first) # [0]
        remaining.remove(first)

        while len(selected) < self.top_k and remaining:
            mmr_scores = []

            for candidate in remaining:
                # Take the relevance to the query for each candidate
                relevance = query_similarities[candidate]

                # Compute the max similarity between already selected images and each candidate
                redundancy = np.max(candidate_similarities[candidate, selected])

                # Compute MMR for each candidate and add the tuple [mrr_score, candidate_index] to the list
                mmr = self.lambda_ * relevance - (1 - self.lambda_) * redundancy
                mmr_scores.append((mmr, candidate))

            _, best_candidate = max(mmr_scores)

            selected.append(best_candidate)
            remaining.remove(best_candidate)

        return {
            "documents": [documents[i] for i in selected]
        }

In [ ]:
class PipelineWrapper(BasePipelineWrapper):
    def setup(self) -> None:
        document_store = get_document_store(Qdrant.COLLECTION_QWEN, Qdrant.EMBEDDING_DIM_QWEN)

        self.retrieval_pipeline = Pipeline()

        self.retrieval_pipeline.add_component("query_embedder", QueryEmbedder(model=VLLM.EMBEDDING_MODEL, api_base_url=VLLM.API_URL, api_key=VLLM.API_KEY))
        self.retrieval_pipeline.add_component("retriever", QdrantEmbeddingRetriever(document_store=document_store, top_k=MMR_CANDIDATES, return_embedding=True))
        self.retrieval_pipeline.add_component("mmr", MMRReranker(top_k=TOP_K, lambda_=0.8))

        self.retrieval_pipeline.connect("query_embedder.embedding", "retriever")
        self.retrieval_pipeline.connect("query_embedder.embedding", "mmr.query_embedding")
        self.retrieval_pipeline.connect("retriever.documents", "mmr.documents")

    def retrieve_documents(self, question: str, images: list[str]):
        retrieval_run_args = {
            "query_embedder": {
                "text": question,
                "image_data_urls": images,
            },
        }

        response = self.retrieval_pipeline.run(retrieval_run_args)

        documents = (response.get("mmr", {}).get("documents", []))
        # documents = (response.get("retriever", {}).get("documents", []))

        return documents

In [ ]:
qwen_retriever = PipelineWrapper()
qwen_retriever.setup()

In [ ]:
def get_crop_path(image_id: str, crops_dir: str | Path = SEMDataset.CROPS) -> Path:
    crops_dir = Path(crops_dir)

    matches = list(crops_dir.glob(f"{image_id}_*.npy"))

    if not matches:
        raise FileNotFoundError(f"No crop found for image ID: {image_id}")

    if len(matches) > 1:
        raise ValueError(f"Multiple crops found for image ID {image_id}: {matches}")

    return matches[0]

In [ ]:
qwen_results = qwen_retriever.retrieve_documents(
    "show sem images of nanoparticles",
    [],
)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(30, 7))
axes = axes.ravel()

for i, result in enumerate(qwen_results[:5]):
    # print(result.id, result.content)

    crop_path = get_crop_path(result.id)
    image = np.load(crop_path)

    axes[i].imshow(image, cmap="gray")
    axes[i].set_title(f"{i+1} - {result.id}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()